# Predict Parent Motivation — Per-Parent LOPO

Does knowing the **parent** (their history) improve motivation prediction beyond the
universal model?

**CV scheme**: Leave-One-Participant-Out (LOPO) — 20 folds.

**Three tiers**:
1. Universal: a-priori structured features only
2. +Population history: training-fold motivation-class proportions (8 features)
3. +Per-participant history: test parent's other highlights' motivation distribution (oracle)

**Primary metric**: weighted F1. Secondary: top-2 accuracy (lenient for 8-class problem).
**Majority baseline**: ~36.6 % (Response Usefulness).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

DATA_DIR   = Path('../../data-exports/20260412_183830')
OUTPUT_DIR = DATA_DIR / 'highlight_analysis_output'
OUT_DIR    = OUTPUT_DIR / 'motivation_classifier_output'
OUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

## Load data

In [ ]:
df_sel = pd.read_csv(OUTPUT_DIR / 'df_sel.csv')

r5 = pd.read_csv(DATA_DIR / 'R5_highlights_coded', sep='\t')
r5.columns = r5.columns.str.strip()
r5 = r5.rename(columns={
    'highlight_id':      'selection_id',
    'Parent Motivation': 'r5_motivation',
    'Model Strategy':    'r5_strategy',
})
r5['r5_motivation'] = r5['r5_motivation'].replace({
    '': None, 'null': None,
    'Response Could Evoke Strong Emotion': 'Response Could Evoke Strong Emotions',
    'Parents Trust of Model Capabilities': None,
})
r5 = r5[r5['r5_motivation'].notna()]

df = df_sel.merge(r5[['selection_id', 'r5_motivation']], on='selection_id', how='inner')
df = df.drop(columns=['parent_motivation'], errors='ignore').rename(
    columns={'r5_motivation': 'parent_motivation'}
).reset_index(drop=True)

print(f'Shape: {df.shape}  |  Participants: {df["prolific_pid"].nunique()}')
print(df['prolific_pid'].value_counts().sort_values().to_string())

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(df['parent_motivation'])
CLASSES = le.classes_
N_CLASSES = len(CLASSES)

print(f'{N_CLASSES} motivation classes:')
for i, (cls, n) in enumerate(zip(CLASSES, np.bincount(y))):
    print(f'  [{i}] {cls}: {n}')


def build_features(df_in: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df_in.index)
    out['strategy_is_null'] = df_in['model_strategy'].isna().astype(int)
    out = pd.concat([out,
        pd.get_dummies(df_in['model_strategy'].fillna('none'), prefix='strat'),
    ], axis=1)
    for col in ['domain', 'age_band', 'sensitivity_level', 'relationship_frame',
                'space_type', 'trait', 'trait_level']:
        out = pd.concat([out,
            pd.get_dummies(df_in[col].fillna('unknown'), prefix=col)
        ], axis=1)
    out['breakdown_expected'] = (df_in['breakdown_expected'] == 'yes').astype(int)
    if 'source' in df_in.columns:
        out = pd.concat([out,
            pd.get_dummies(df_in['source'].fillna('unknown'), prefix='source')
        ], axis=1)
    for col in ['parent_gender', 'parent_age_group', 'parent_education', 'parent_ethnicity',
                'area_of_residency', 'child_has_ai_use', 'parent_llm_monitoring_level']:
        out = pd.concat([out,
            pd.get_dummies(df_in[col].fillna('unknown'), prefix=col)
        ], axis=1)
    out['genai_regular_user'] = (df_in['genai_familiarity'] == 'regular_user').astype(int)
    freq_map = {'daily': 2, 'weekly': 1, 'monthly_or_less': 0}
    out['genai_usage_freq'] = df_in['genai_usage_frequency'].map(freq_map).fillna(0).astype(int)
    out['parent_internet_use_frequency'] = pd.to_numeric(
        df_in['parent_internet_use_frequency'], errors='coerce').fillna(0)
    out['is_only_child'] = (df_in['is_only_child'].astype(str).str.lower() == 'yes').astype(int)
    for s in 'ABCD':
        out[f'parenting_{s}'] = df_in['parenting_style'].fillna('').str.contains(s).astype(int)
    return out.astype(float)


X_struct = build_features(df)
print(f'\nStructured features: {X_struct.shape[1]}')

In [ ]:
def top2_acc(y_true: np.ndarray, y_proba: np.ndarray) -> float:
    top2 = np.argsort(y_proba, axis=1)[:, -2:]
    return float(np.mean([y_true[i] in top2[i] for i in range(len(y_true))]))


LOPO_MODELS = {
    'Majority Baseline':   DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=300, max_depth=8,
                                                   random_state=RANDOM_STATE),
}

## Tier 1 — Universal LOPO (structured baseline)

In [ ]:
def run_lopo_structured(df, X, y, models: dict) -> pd.DataFrame:
    rows = []
    for pid in df['prolific_pid'].unique():
        test_mask = (df['prolific_pid'] == pid).values
        X_tr, X_te = X[~test_mask], X[test_mask]
        y_tr, y_te = y[~test_mask], y[test_mask]

        for name, model in models.items():
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            f1w = f1_score(y_te, y_pred, average='weighted', zero_division=0)
            t2a = np.nan
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(X_te)
                if proba.shape[1] == N_CLASSES:
                    t2a = top2_acc(y_te, proba)
            rows.append({
                'prolific_pid': pid, 'n_test': test_mask.sum(), 'model': name,
                'f1_weighted':  round(f1w, 3),
                'top2_acc':     round(t2a, 3) if not np.isnan(t2a) else np.nan,
            })
    return pd.DataFrame(rows)


lopo_universal = run_lopo_structured(df, X_struct.values, y, LOPO_MODELS)
print('Tier 1 — Universal LOPO (mean across participants):')
print(lopo_universal.groupby('model')[['f1_weighted', 'top2_acc']].mean().round(3).to_string())
lopo_universal.to_csv(OUT_DIR / 'motivation_per_parent_lopo_universal.csv', index=False)

## Tier 2 — Universal + population history (8 class-proportion features)

For each LOPO fold, compute the proportion of each motivation class in the training set
and broadcast as features to the test rows.  These are 8 scalar features representing
the population base-rate for each motivation class.

In [ ]:
def build_pop_history(y_train: np.ndarray, n_test: int, n_classes: int) -> np.ndarray:
    counts = np.bincount(y_train, minlength=n_classes)
    props  = counts / counts.sum()
    return np.tile(props, (n_test, 1))


def run_lopo_with_pop_history(df, X_struct, y, models: dict) -> pd.DataFrame:
    rows = []
    for pid in df['prolific_pid'].unique():
        test_mask = (df['prolific_pid'] == pid).values
        y_tr, y_te = y[~test_mask], y[test_mask]

        pop_tr = build_pop_history(y_tr, (~test_mask).sum(), N_CLASSES)
        pop_te = build_pop_history(y_tr,   test_mask.sum(),  N_CLASSES)
        X_tr = np.hstack([X_struct[~test_mask], pop_tr])
        X_te = np.hstack([X_struct[test_mask],  pop_te])

        for name, model in models.items():
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            f1w = f1_score(y_te, y_pred, average='weighted', zero_division=0)
            t2a = np.nan
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(X_te)
                if proba.shape[1] == N_CLASSES:
                    t2a = top2_acc(y_te, proba)
            rows.append({
                'prolific_pid': pid, 'n_test': test_mask.sum(), 'model': name,
                'f1_weighted':  round(f1w, 3),
                'top2_acc':     round(t2a, 3) if not np.isnan(t2a) else np.nan,
            })
    return pd.DataFrame(rows)


lopo_history = run_lopo_with_pop_history(df, X_struct.values, y, LOPO_MODELS)
print('Tier 2 — +Population history (mean across participants):')
print(lopo_history.groupby('model')[['f1_weighted', 'top2_acc']].mean().round(3).to_string())
lopo_history.to_csv(OUT_DIR / 'motivation_per_parent_lopo_history.csv', index=False)

## Tier 3 — Universal + per-participant history (oracle)

For each test row of the held-out parent, compute the motivation-class distribution
from that parent's *other* test rows.  This is oracle information (uses test-set labels)
but establishes an upper bound on what within-parent consistency provides.

In [ ]:
def build_pp_history(df_rows: pd.DataFrame, y_rows: np.ndarray,
                     pop_fallback: np.ndarray, n_classes: int) -> np.ndarray:
    """For each row, use all other rows of the same pid as history. Fallback = pop."""
    result = []
    for i in range(len(df_rows)):
        pid    = df_rows.iloc[i]['prolific_pid']
        others = [j for j in range(len(df_rows))
                  if j != i and df_rows.iloc[j]['prolific_pid'] == pid]
        if not others:
            result.append(pop_fallback)
        else:
            counts = np.bincount(y_rows[others], minlength=n_classes)
            result.append(counts / counts.sum())
    return np.array(result)


def run_lopo_with_pp_history(df, X_struct, y, models: dict) -> pd.DataFrame:
    rows = []
    for pid in df['prolific_pid'].unique():
        test_mask = (df['prolific_pid'] == pid).values
        df_tr     = df[~test_mask].reset_index(drop=True)
        df_te     = df[test_mask].reset_index(drop=True)
        y_tr, y_te = y[~test_mask], y[test_mask]

        pop_fb = np.bincount(y_tr, minlength=N_CLASSES) / len(y_tr)

        pp_tr = build_pp_history(df_tr, y_tr, pop_fb, N_CLASSES)
        pp_te = build_pp_history(df_te, y_te, pop_fb, N_CLASSES)

        X_tr = np.hstack([X_struct[~test_mask], pp_tr])
        X_te = np.hstack([X_struct[test_mask],  pp_te])

        for name, model in models.items():
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            f1w = f1_score(y_te, y_pred, average='weighted', zero_division=0)
            t2a = np.nan
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(X_te)
                if proba.shape[1] == N_CLASSES:
                    t2a = top2_acc(y_te, proba)
            rows.append({
                'prolific_pid': pid, 'n_test': test_mask.sum(), 'model': name,
                'f1_weighted':  round(f1w, 3),
                'top2_acc':     round(t2a, 3) if not np.isnan(t2a) else np.nan,
            })
    return pd.DataFrame(rows)


lopo_pp = run_lopo_with_pp_history(df, X_struct.values, y, LOPO_MODELS)
print('Tier 3 — +Per-participant history (mean across participants):')
print(lopo_pp.groupby('model')[['f1_weighted', 'top2_acc']].mean().round(3).to_string())
lopo_pp.to_csv(OUT_DIR / 'motivation_per_parent_lopo_per_participant.csv', index=False)

## Summary comparison

In [ ]:
def agg(df_lopo, model='Random Forest'):
    sub = df_lopo[df_lopo['model'] == model]
    return sub[['f1_weighted', 'top2_acc']].mean().round(3)

summary_rows = []
for tier, lopo_df in [
    ('Universal',         lopo_universal),
    ('+Pop history',      lopo_history),
    ('+Per-participant',  lopo_pp),
]:
    for model in ['Majority Baseline', 'Logistic Regression', 'Random Forest']:
        sub = lopo_df[lopo_df['model'] == model]
        summary_rows.append({
            'tier':        tier,
            'model':       model,
            'f1_weighted': round(sub['f1_weighted'].mean(), 3),
            'top2_acc':    round(sub['top2_acc'].mean(), 3),
        })

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))
summary.to_csv(OUT_DIR / 'motivation_per_parent_summary.csv', index=False)

## Per-parent F1 comparison plot

In [ ]:
# Merge the 3 tiers on prolific_pid for RF
u_rf = lopo_universal[lopo_universal['model'] == 'Random Forest'][['prolific_pid', 'n_test', 'f1_weighted']]
h_rf = lopo_history[lopo_history['model'] == 'Random Forest'][['prolific_pid', 'f1_weighted']]
p_rf = lopo_pp[lopo_pp['model'] == 'Random Forest'][['prolific_pid', 'f1_weighted']]

cmp = (
    u_rf.rename(columns={'f1_weighted': 'f1_universal'})
    .merge(h_rf.rename(columns={'f1_weighted': 'f1_pop'}), on='prolific_pid')
    .merge(p_rf.rename(columns={'f1_weighted': 'f1_pp'}), on='prolific_pid')
    .sort_values('n_test')
)

x, w = np.arange(len(cmp)), 0.25
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w,   cmp['f1_universal'], w, label='Universal RF',      color='steelblue')
ax.bar(x,       cmp['f1_pop'],       w, label='+Pop history',      color='darkorange')
ax.bar(x + w,   cmp['f1_pp'],        w, label='+Per-participant',  color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(
    [f'{pid[:6]}\n(n={n})' for pid, n in zip(cmp['prolific_pid'], cmp['n_test'])],
    fontsize=7,
)
ax.set_ylabel('Weighted F1')
ax.set_title('Per-parent motivation prediction: 3 tiers (Random Forest, LOPO)')
ax.axhline(0.366, color='gray', linestyle='--', linewidth=0.8, label='Majority baseline')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'motivation_per_parent_f1_comparison.png', dpi=100)
plt.show()